#Dimension Table - Deaths

> ## 1. View Data First

In [0]:
from pyspark.sql import functions as F

In [0]:
deaths = spark.table("himalaya.silver.deaths")
peaks = spark.table("himalaya.silver.expeditions_peaks")
members = spark.table("himalaya.silver.expeditions_members")

> ##2. Drop Ingestion

  _not needed in gold_

In [0]:
deaths = deaths.drop("ingested_at")

>##3. Get Peak ID
_Joining with dim_peaks with the mountain name_

In [0]:
deaths = deaths.withColumn("mountain",
    F.when(F.col("mountain") == "Mount Everest", "Everest")
     .otherwise(F.col("mountain"))
)

In [0]:
deaths_source = deaths.join(peaks, deaths["mountain"] == peaks["peak_name"], "left") \
    .select(
        deaths["name"],
        deaths["date"],
        deaths["cause_of_death_category"],
        deaths["mountain"],
        peaks["peakid"]
    )

In [0]:
# Filter out unmatched peaks
deaths_source = deaths_source.filter(F.col("peakid").isNotNull())

>##4. Get Deaths from members

In [0]:
members_deaths = members.filter(F.col("death") == True) \
    .select("name", "peakid", "expid", "death_date")

>##5. Full outer join to capture deaths from both sources

In [0]:
df = deaths_source.join(members_deaths,
    (deaths_source["name"] == members_deaths["name"]) &
    (deaths_source["peakid"] == members_deaths["peakid"]), "full") \
    .select(
        F.coalesce(deaths_source["name"], members_deaths["name"]).alias("name"),
        F.coalesce(deaths_source["peakid"], members_deaths["peakid"]).alias("peakid"),
        deaths_source["mountain"],
        deaths_source["cause_of_death_category"],
        members_deaths["expid"],
        members_deaths["death_date"]
    )

>##_Quick Null fixes_

In [0]:
# Filter nulls from members side of full join
df = df.filter(F.col("peakid").isNotNull())

In [0]:
df = df.join(peaks.select("peakid", "peak_name"), "peakid", "left") \
    .withColumn("mountain", F.coalesce(F.col("mountain"), F.col("peak_name"))) \
    .drop("peak_name")

In [0]:
df = df.withColumn("cause_of_death_category", 
    F.coalesce(F.col("cause_of_death_category"), F.lit("Unknown"))
)

>##_Reorder_

In [0]:
df = df.select("expid", "peakid", "name", "mountain", "cause_of_death_category", "death_date")

>##6. Write to Gold

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("himalaya.gold.dim_deaths")
print("✅ Written to himalaya.gold.dim_deaths")

In [0]:
display(spark.table("himalaya.gold.dim_deaths").limit(5))